# AkwaabaFit Food Detection — YOLOv8 Training

This notebook trains a YOLOv8n model on the merged food dataset using Google Colab's free T4 GPU.

**Steps:**
1. Mount Google Drive (your `food_dataset.zip` should be in `My Drive`)
2. Extract dataset & install dependencies
3. Train the model
4. Export to ONNX
5. Copy results back to Drive for download

In [ ]:
# Step 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2 — Install ultralytics
!pip install -q ultralytics

In [ ]:
# Step 3 — Check GPU
!nvidia-smi

In [ ]:
# Step 4 — Extract dataset from Drive to Colab local disk (faster I/O)
import zipfile, os, time

ZIP_PATH = '/content/drive/MyDrive/food_dataset.zip'
EXTRACT_DIR = '/content/food_dataset'

if not os.path.exists(EXTRACT_DIR):
    print(f'Extracting {ZIP_PATH} ...')
    start = time.time()
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print(f'Done in {time.time()-start:.0f}s')
else:
    print('Dataset already extracted.')

!ls /content/food_dataset/

In [ ]:
# Step 5 — Fix data.yaml paths for Colab
import yaml

YAML_PATH = '/content/food_dataset/data.yaml'

with open(YAML_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['path'] = '/content/food_dataset'
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images'
cfg['test'] = 'test/images'

with open(YAML_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"Classes: {cfg['nc']}")
print(f"Path: {cfg['path']}")
print('data.yaml updated for Colab paths.')

In [ ]:
# Step 6 — Train YOLOv8n
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=YAML_PATH,
    epochs=100,
    imgsz=640,
    batch=16,
    project='/content/runs',
    name='food_model',
    exist_ok=True,
    device=0,
    patience=20,
    workers=2,
    plots=True,
    amp=True,
)

In [ ]:
# Step 7 — Validate
best = YOLO('/content/runs/food_model/weights/best.pt')
metrics = best.val(data=YAML_PATH, imgsz=640)
print(f"\nmAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

In [ ]:
# Step 8 — Export to ONNX
best.export(format='onnx', imgsz=640)
print('ONNX model exported.')

In [ ]:
# Step 9 — Copy results back to Google Drive
import shutil

DRIVE_OUT = '/content/drive/MyDrive/food_model_results'
os.makedirs(DRIVE_OUT, exist_ok=True)

for f in ['best.pt', 'best.onnx', 'last.pt']:
    src = f'/content/runs/food_model/weights/{f}'
    if os.path.exists(src):
        shutil.copy2(src, DRIVE_OUT)
        print(f'Copied {f} to Drive')

for f in ['results.csv', 'results.png', 'confusion_matrix.png']:
    src = f'/content/runs/food_model/{f}'
    if os.path.exists(src):
        shutil.copy2(src, DRIVE_OUT)
        print(f'Copied {f} to Drive')

print(f'\nAll results saved to Google Drive: {DRIVE_OUT}')
print('You can download best.onnx from there.')